
# 2) Information Extraction


Tennis Grand Slam Knowledge Graph and Intelligent QA System (RAG) 

Step 2: Information Extraction


## Extraction Strategy
We use two relation-extraction strategies:

1. **Sentence-level co-occurrence**
   This strategy looks for domain cues such as *won*, *surface*, or *from* inside a sentence. It is simple and useful for a first KB prototype.

2. **Dependency-oriented extraction**
   This strategy uses the dependency parse produced by spaCy to capture more explicit player-opponent relations such as *X defeated Y*.

The combination is still imperfect, but it is a strong pedagogical baseline before moving to RDF modeling.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import spacy
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.ie.information_extraction import (
    GENERIC_ENTITY_TERMS,
    compute_entity_type_distribution,
    count_filtered_generic_candidates,
    load_spacy_model,
    manual_review_sample,
    noisy_triple_ratio,
    normalize_entity_name,
    process_corpus,
    relation_distribution,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "crawler_output.jsonl"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
ENTITIES_PATH = INTERIM_DIR / "extracted_entities.csv"
TRIPLES_PATH = INTERIM_DIR / "extracted_knowledge.csv"

print(f"Raw corpus path: {RAW_PATH}")
print(f"Entities output path: {ENTITIES_PATH}")
print(f"Triples output path: {TRIPLES_PATH}")


Raw corpus path: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/raw/crawler_output.jsonl
Entities output path: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/interim/extracted_entities.csv
Triples output path: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/interim/extracted_knowledge.csv



## Load the Cleaned JSON Data
We reuse the output of Notebook 1. This keeps the project modular: the crawler is responsible for document acquisition and cleaning, while this notebook is responsible for turning clean text into structured candidate knowledge.


In [2]:

with RAW_PATH.open("r", encoding="utf-8") as file:
    records = [json.loads(line) for line in file]

corpus_df = pd.DataFrame(records)
print(f"Loaded {len(corpus_df)} cleaned documents.")
display(corpus_df[["title", "url", "word_count"]].head())


Loaded 11 cleaned documents.


,title,url,word_count
0,Grand Slam (tennis),https://en.wikipedia.org/wiki/Grand_Slam_(tennis),9685
1,Australian Open,https://en.wikipedia.org/wiki/Australian_Open,5528
2,French Open,https://en.wikipedia.org/wiki/French_Open,5011
3,Wimbledon Championships,https://en.wikipedia.org/wiki/Wimbledon_Champi...,13670
4,US Open (tennis),https://en.wikipedia.org/wiki/US_Open_(tennis),5656



## Why We Use `en_core_web_trf`
The `en_core_web_trf` model is a transformer-based English pipeline for spaCy. It includes:
- a transformer encoder;
- a part-of-speech tagger;
- a dependency parser;
- a lemmatizer;
- a named entity recognizer.

This makes it suitable for our task because we need both NER and sentence structure. It is heavier than smaller models, but it usually gives better results on complex pages.


In [3]:
nlp = load_spacy_model("en_core_web_trf")
print("Loaded spaCy model successfully.")
print("Pipeline components:", nlp.pipe_names)

Loaded spaCy model successfully.
Pipeline components: ['transformer', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']



## Quick NER Demonstration
Before processing the full corpus, it is good practice to inspect one small example manually. This helps us verify that the model recognizes tennis players, tournaments, places, and dates in a plausible way.


In [ ]:
sample_text = corpus_df.loc[0, "text"][:1200]
doc = nlp(sample_text)
sample_entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ in {"PERSON", "ORG", "GPE", "DATE"}]
print(sample_entities[:25])

[('a calendar year', 'DATE'), ('the same calendar year', 'DATE'), ('non-calendar-year', 'DATE'), ('annual', 'DATE'), ('recent years', 'DATE'), ('the International Tennis Federation', 'ORG'), ('the Association of Tennis Professionals', 'ORG'), ('ATP', 'ORG'), ("Women's Tennis Association", 'ORG'), ('WTA', 'ORG'), ('ATP', 'ORG'), ('WTA', 'ORG')]



## Domain-Specific Filtering and Normalization
Raw NER output is not enough. We also need to reduce domain noise.

### Why filtering?
Some strings are too generic to become useful KG nodes. Examples include:
- `Open`
- `Final`
- `Championship`
- `Seed`
- `Court`

If these appear alone, they are ambiguous and often not useful.

### Why normalization?
A tennis corpus contains variants such as:
- `Roland Garros` and `French Open`
- `Wimbledon` and `Wimbledon Championships`
- `US Open` and `U.S. Open`

Normalization avoids creating multiple graph nodes for the same real-world concept.


In [ ]:
normalization_examples = pd.DataFrame(
    {
        "raw_form": ["Roland Garros", "Roland-Garros", "Wimbledon", "U.S. Open", "Novak Djokovic"],
        "normalized_form": [
            normalize_entity_name("Roland Garros"),
            normalize_entity_name("Roland-Garros"),
            normalize_entity_name("Wimbledon"),
            normalize_entity_name("U.S. Open"),
            normalize_entity_name("Novak Djokovic"),
        ],
    }
)
display(normalization_examples)
print("Generic entity terms filtered when isolated:", sorted(GENERIC_ENTITY_TERMS))

,raw_form,normalized_form
0,Roland Garros,French Open
1,Roland-Garros,French Open
2,Wimbledon,Wimbledon Championships
3,U.S. Open,US Open
4,Novak Djokovic,Novak Djokovic


Generic entity terms filtered when isolated: ['championship', 'championships', 'court', 'final', 'open', 'seed']



## Run Information Extraction on the Corpus
The next cell executes the full pipeline:
- text chunking to keep transformer processing manageable;
- entity extraction;
- normalization and filtering;
- candidate relation extraction;
- CSV export.


In [ ]:
entities_df, triples_df = process_corpus(records, nlp)
entities_df.to_csv(ENTITIES_PATH, index=False)
triples_df.to_csv(TRIPLES_PATH, index=False)

print(f"Saved {len(entities_df)} entities to {ENTITIES_PATH}")
print(f"Saved {len(triples_df)} triples to {TRIPLES_PATH}")

Saved 13464 entities to /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/interim/extracted_entities.csv
Saved 722 triples to /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/interim/extracted_knowledge.csv



## Inspect the Extracted Entities
We first inspect the entity table. This helps us verify the effect of normalization and see whether the main entity families are present.


In [ ]:
display(entities_df.head(15))
print(f"Unique normalized entities: {entities_df['normalized_entity'].nunique()}")

,entity_text,entity_label,normalized_entity,source_url,sentence
0,"""The All England Croquet and Lawn Tennis Club",ORG,"""The All England Croquet and Lawn Tennis Club",https://en.wikipedia.org/wiki/Wimbledon_Champi...,"In spring 1877, the club was renamed ""The All ..."
1,1 April 2016,DATE,1 April 2016,https://en.wikipedia.org/wiki/Australian_Open,Archived from the original (PDF) on 1 April 2016.
2,1 April 2019,DATE,1 April 2019,https://en.wikipedia.org/wiki/Novak_Djokovic,Retrieved 1 April 2019.
3,1 April 2020,DATE,1 April 2020,https://en.wikipedia.org/wiki/Wimbledon_Champi...,"As a result of the COVID-19 global pandemic, t..."
4,1 April 2020,DATE,1 April 2020,https://en.wikipedia.org/wiki/Wimbledon_Champi...,Retrieved 1 April 2020.
5,1 April 2023,DATE,1 April 2023,https://en.wikipedia.org/wiki/Carlos_Alcaraz,1 April 2023.
6,1 April 2023,DATE,1 April 2023,https://en.wikipedia.org/wiki/Novak_Djokovic,1 April 2023.
7,1 April 2023,DATE,1 April 2023,https://en.wikipedia.org/wiki/Rafael_Nadal,Archived from the original on 1 April 2023.
8,1 April 2024,DATE,1 April 2024,https://en.wikipedia.org/wiki/Carlos_Alcaraz,Retrieved 1 April 2024.
9,1 August 2008,DATE,1 August 2008,https://en.wikipedia.org/wiki/Rafael_Nadal,1 August 2008.


Unique normalized entities: 5802



## Entity Type Distribution
This distribution gives a first picture of the extraction profile. In our domain, we expect many `PERSON` mentions, but also many `DATE` mentions because sports articles frequently mention years and tournament editions.


In [ ]:
entity_type_df = compute_entity_type_distribution(entities_df)
display(entity_type_df)

,entity_type,count
0,PERSON,6441
1,DATE,4663
2,ORG,1503
3,GPE,857



## Inspect the Candidate Triples
Each triple candidate contains:
- `subject`
- `relation`
- `object`
- `sentence`
- `source_url`
- `strategy`

This traceability is essential because it lets us review errors and justify later KG decisions.


In [9]:
display(triples_df.head(20))

,subject,subject_type,subject_role,relation,object,object_type,object_role,sentence,source_url,strategy
0,Agassi,PERSON,Player,won,Wimbledon Championships,ORG,Tournament,"Following his Wimbledon loss, Nadal won 16 con...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
1,Albert Montañés,PERSON,Player,playedAgainst,Albert Portas,PERSON,Player,"In October, Nadal defeated No. 76 Albert Monta...",https://en.wikipedia.org/wiki/Rafael_Nadal,dependency
2,Alcaraz,PERSON,Player,participatedIn,French Open,ORG,Tournament,"In September 2020, aged 17, Alcaraz played his...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,cooccurrence
3,Alcaraz,PERSON,Player,participatedIn,Grand Slam,ORG,Tournament,"In doing so, Alcaraz also broke Nadal's record...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
4,Alcaraz,PERSON,Player,playedAgainst,Alex de Minaur,PERSON,Player,"At the Queen's Club, Alcaraz claimed his first...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
5,Alcaraz,PERSON,Player,playedAgainst,Alexander Zverev,PERSON,Player,Alcaraz defeated Alexander Zverev in another f...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
6,Alcaraz,PERSON,Player,playedAgainst,Botic van de Zandschulp,PERSON,Player,In his first main draw match at the Australian...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
7,Alcaraz,PERSON,Player,playedAgainst,Cameron Norrie,PERSON,Player,"At Indian Wells, Alcaraz defeated defending ch...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
8,Alcaraz,PERSON,Player,playedAgainst,Casper Ruud,PERSON,Player,"Two weeks later, at the Miami Open, Alcaraz de...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
9,Alcaraz,PERSON,Player,playedAgainst,Djokovic,PERSON,Player,The duo soon met for a rematch in the 2023 Wim...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency



## Relation Distribution
This table shows which relations are most frequent in the extracted candidate knowledge. It helps us detect imbalance, such as an excessive number of opponent relations compared with winner relations.


In [10]:
relation_df = relation_distribution(triples_df)
display(relation_df)

,relation,count
0,playedAgainst,433
1,won,170
2,hasSurface,69
3,participatedIn,31
4,fromCountry,19



## Quality Control
Extraction quality should never be assumed. We therefore check:
- the total number of entities;
- the type distribution;
- the total number of triples;
- a rough noisy-triple proportion in a manual sample;
- examples of filtered-out generic entities.


In [11]:

raw_doc = nlp(corpus_df.loc[0, 'text'][:3000])
raw_entities = [ent.text for ent in raw_doc.ents if ent.label_ in {"PERSON", "ORG", "GPE", "DATE"}]
filtered_generic_counter = count_filtered_generic_candidates(raw_entities)

quality_summary = {
    "total_entities_extracted": int(len(entities_df)),
    "unique_normalized_entities": int(entities_df['normalized_entity'].nunique()),
    "total_triples_extracted": int(len(triples_df)),
    "sample_noisy_triple_ratio": round(noisy_triple_ratio(triples_df, sample_size=30), 3),
    "filtered_generic_examples": dict(filtered_generic_counter.most_common(10)),
}

print(json.dumps(quality_summary, indent=2, ensure_ascii=False))


{
  "total_entities_extracted": 13464,
  "unique_normalized_entities": 5802,
  "total_triples_extracted": 722,
  "sample_noisy_triple_ratio": 0.0,
  "filtered_generic_examples": {
    "US": 2
  }
}



## Manual Inspection Sample
Automatic metrics are not enough. In information extraction, a short manual inspection is necessary because many errors are semantic rather than syntactic.


In [12]:
review_sample_df = manual_review_sample(triples_df, sample_size=12)
display(review_sample_df)

,subject,subject_type,subject_role,relation,object,object_type,object_role,sentence,source_url,strategy
0,Agassi,PERSON,Player,won,Wimbledon Championships,ORG,Tournament,"Following his Wimbledon loss, Nadal won 16 con...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
1,Albert Montañés,PERSON,Player,playedAgainst,Albert Portas,PERSON,Player,"In October, Nadal defeated No. 76 Albert Monta...",https://en.wikipedia.org/wiki/Rafael_Nadal,dependency
2,Alcaraz,PERSON,Player,participatedIn,French Open,ORG,Tournament,"In September 2020, aged 17, Alcaraz played his...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,cooccurrence
3,Alcaraz,PERSON,Player,participatedIn,Grand Slam,ORG,Tournament,"In doing so, Alcaraz also broke Nadal's record...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
4,Alcaraz,PERSON,Player,playedAgainst,Alex de Minaur,PERSON,Player,"At the Queen's Club, Alcaraz claimed his first...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
5,Alcaraz,PERSON,Player,playedAgainst,Alexander Zverev,PERSON,Player,Alcaraz defeated Alexander Zverev in another f...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
6,Alcaraz,PERSON,Player,playedAgainst,Botic van de Zandschulp,PERSON,Player,In his first main draw match at the Australian...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
7,Alcaraz,PERSON,Player,playedAgainst,Cameron Norrie,PERSON,Player,"At Indian Wells, Alcaraz defeated defending ch...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
8,Alcaraz,PERSON,Player,playedAgainst,Casper Ruud,PERSON,Player,"Two weeks later, at the Miami Open, Alcaraz de...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
9,Alcaraz,PERSON,Player,playedAgainst,Djokovic,PERSON,Player,The duo soon met for a rematch in the 2023 Wim...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency



## Correct and Incorrect Extraction Examples
A serious notebook should show both successful and problematic examples.

### Typical correct examples
We usually obtain plausible facts such as:
- player vs player opponent relations from sentences containing *defeated* or *beat*;
- player-country relations when the sentence explicitly says *from X*;
- tournament-surface relations when the tournament name and a surface mention appear in the same sentence.

### Typical incorrect examples
Errors still happen, for example when:
- a long historical sentence contains many names and only one real tennis fact;
- a quote headline is treated like a factual sentence;
- a generic label such as `Open Era` looks like a tournament even though it is not one.

These limitations are important because they justify why Notebook 3 will include stronger RDF validation and why later alignment steps must be careful.


In [13]:

correct_examples = triples_df[
    triples_df['sentence'].str.contains('defeated|beat|from|grass|clay', case=False, regex=True, na=False)
].head(8)

incorrect_like_examples = triples_df[
    triples_df['sentence'].str.contains('Open Era|archived|final for the ages|record', case=False, regex=True, na=False)
].head(8)

print('Potentially correct examples:')
display(correct_examples)
print('Potentially noisy examples to review:')
display(incorrect_like_examples)


Potentially correct examples:


,subject,subject_type,subject_role,relation,object,object_type,object_role,sentence,source_url,strategy
1,Albert Montañés,PERSON,Player,playedAgainst,Albert Portas,PERSON,Player,"In October, Nadal defeated No. 76 Albert Monta...",https://en.wikipedia.org/wiki/Rafael_Nadal,dependency
4,Alcaraz,PERSON,Player,playedAgainst,Alex de Minaur,PERSON,Player,"At the Queen's Club, Alcaraz claimed his first...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
5,Alcaraz,PERSON,Player,playedAgainst,Alexander Zverev,PERSON,Player,Alcaraz defeated Alexander Zverev in another f...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
6,Alcaraz,PERSON,Player,playedAgainst,Botic van de Zandschulp,PERSON,Player,In his first main draw match at the Australian...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
7,Alcaraz,PERSON,Player,playedAgainst,Cameron Norrie,PERSON,Player,"At Indian Wells, Alcaraz defeated defending ch...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
8,Alcaraz,PERSON,Player,playedAgainst,Casper Ruud,PERSON,Player,"Two weeks later, at the Miami Open, Alcaraz de...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
9,Alcaraz,PERSON,Player,playedAgainst,Djokovic,PERSON,Player,The duo soon met for a rematch in the 2023 Wim...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
10,Alcaraz,PERSON,Player,playedAgainst,Djokovic,PERSON,Player,They would meet again soon after in the 2023 W...,https://en.wikipedia.org/wiki/Novak_Djokovic,dependency


Potentially noisy examples to review:


,subject,subject_type,subject_role,relation,object,object_type,object_role,sentence,source_url,strategy
3,Alcaraz,PERSON,Player,participatedIn,Grand Slam,ORG,Tournament,"In doing so, Alcaraz also broke Nadal's record...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
10,Alcaraz,PERSON,Player,playedAgainst,Djokovic,PERSON,Player,They would meet again soon after in the 2023 W...,https://en.wikipedia.org/wiki/Novak_Djokovic,dependency
19,Alcaraz,PERSON,Player,playedAgainst,Marin Čilić,PERSON,Player,"In the fourth round, Alcaraz defeated former c...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
47,Alexander Zverev,PERSON,Player,playedAgainst,McEnroe,PERSON,Player,"At Rome, Nadal won his 8th title beating Alexa...",https://en.wikipedia.org/wiki/Rafael_Nadal,dependency
51,Alexander Zverev,PERSON,Player,won,Open Era,ORG,Tournament,"At Rome, Nadal won his 8th title beating Alexa...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
53,Andre Agassi's,PERSON,Player,won,Open Era,ORG,Tournament,"He won 24 consecutive singles matches, breakin...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
83,Cameron Norrie,PERSON,Player,playedAgainst,Nick Kyrgios,PERSON,Player,He reached a record 32nd Grand Slam final afte...,https://en.wikipedia.org/wiki/Novak_Djokovic,dependency
85,Cameron Norrie,PERSON,Player,playedAgainst,Roger Federer,PERSON,Player,He reached a record 32nd Grand Slam final afte...,https://en.wikipedia.org/wiki/Novak_Djokovic,dependency



## Ambiguity Analysis
Below are three important ambiguity cases from the tennis domain.

`Open`
The word `Open` alone is too generic. It could refer to several tournaments. Therefore, we only keep normalized full forms such as `Australian Open` or `US Open`.

`Final`
The word `Final` is an event stage, not a stable named entity. If kept as a standalone node, it would create weak graph semantics.

`French Open` vs `Roland Garros`
These two names refer to the same tournament. If we do not normalize them, we risk splitting one real entity into two graph nodes.

Player name variants

Sports text may use `Nadal`, `Rafa Nadal`, or `Rafael Nadal`. A full production system would need stronger alias resolution. In this notebook we start with a small manual normalization dictionary.
